# Rfam Tools

![Rfam](https://proto-bio.github.io/proto-assets/images/tool/rfam/hero.png)

This notebook demonstrates the two Rfam wrappers provided by proto_tools. `run_rfam_regions` lists where an RNA family is annotated in sequenced genomes, with coordinates and strand, and `run_rfam_family` fetches a family's curation record and consensus secondary structure. The running example is TwoAYGGAY (RF01731), a bacterial regulatory RNA whose two hairpins carry the AYGGAY motif, in *Pseudomonas fluorescens* SBW25.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("rfam")
display_overview("rfam")
display_docs_section("rfam", "Background")

# Rfam

[Rfam](https://rfam.org) is a curated database of non-coding RNA families. Each family is a seed alignment of representative sequences, a consensus secondary structure, and a covariance model that annotates the family across sequenced genomes. This toolkit wraps two Rfam endpoints: `rfam-family` (a family's curation record, cutoffs, and consensus structure) and `rfam-regions` (every genome region the family is annotated in, with coordinates and strand).

Rfam is described in the Rfam 15 database report ([Ontiveros-Palacios et al., 2025](https://doi.org/10.1093/nar/gkae1023)), published in *Nucleic Acids Research*. It is maintained at [EMBL-EBI](https://www.ebi.ac.uk/) and released in numbered versions; each family records its authors, the source of its seed alignment and structure, and the bit-score cutoffs that define membership. Families with related structures are grouped into clans. Full-region annotations come from searching each family's covariance model against a genome database with [Infernal](https://eddylab.org/infernal/).

Internally, both tools issue HTTP GET requests to the Rfam website under `https://rfam.org/family/<family>`. `rfam-family` reads the family record as JSON and the seed alignment as Stockholm, from which it extracts the consensus structure (`#=GC SS_cons`) and the reference consensus sequence (`#=GC RF`). `rfam-regions` reads the family's regions table as plain text and normalizes each hit so that `start <= end`, with the strand stated separately. Results reflect the current Rfam release, which the outputs report.

## Available tools

In [2]:
display_available_tools("rfam")

- **`run_rfam_family()`** — Fetch an Rfam RNA family's curation record, consensus structure, and seed alignment
- **`run_rfam_regions()`** — List where an Rfam RNA family is annotated in genomes, with coordinates and strand

### `run_rfam_regions`

`run_rfam_regions` lists every sequence region a family is annotated in. Each hit carries its sequence accession, bit score, 1-indexed inclusive coordinates with `start <= end`, strand, species, and NCBI taxonomy ID. Filters narrow the list to one taxonomy ID, a species substring, or a sequence accession. Here we list the TwoAYGGAY hits in *P. fluorescens* SBW25 (taxonomy ID 216595).

In [3]:
from proto_tools import (
    RfamRegionsConfig,
    RfamRegionsInput,
    run_rfam_regions,
)

In [4]:
# Display docs
display_api_reference("rfam", "input", "run_rfam_regions")

# TwoAYGGAY hits in Pseudomonas fluorescens SBW25
regions_inputs = RfamRegionsInput(family="RF01731", taxid=216595)

**Input** — `RfamRegionsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>family</code> | <code>str</code> | required | Rfam accession (e.g. 'RF01731') or family ID (e.g. 'TwoAYGGAY') |
| <code>taxid</code> | <code>int &#124; None</code> | <code>None</code> | Keep only hits in this NCBI taxonomy ID (e.g. 216595) |
| <code>species</code> | <code>str &#124; None</code> | <code>None</code> | Keep only hits whose species contains this text (case-insensitive) |
| <code>sequence_accession</code> | <code>str &#124; None</code> | <code>None</code> | Keep only hits on this sequence; version optional ('AM181176' or 'AM181176.4') |

In [5]:
# Display config docs
display_api_reference("rfam", "config", "run_rfam_regions")

# Defaults return up to 500 regions | see docs above for all fields
regions_config = RfamRegionsConfig()

**Config** — `RfamRegionsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>max_regions</code> | <code>int</code> | <code>500</code> | Most regions returned after filtering; the rest are counted, not listed |

In [6]:
# Run the lookup
regions_result = run_rfam_regions(regions_inputs, regions_config)

In [7]:
# Display output docs
display_api_reference("rfam", "output", "run_rfam_regions")

print(f"{regions_result.rfam_id} ({regions_result.accession}), Rfam {regions_result.rfam_release}")
print(f"{regions_result.matched_regions} of {regions_result.total_regions} regions are in SBW25")
for region in sorted(regions_result.regions, key=lambda r: -r.bit_score)[:5]:
    print(f"  {region.sequence_accession} {region.start}-{region.end} ({region.strand})  {region.bit_score} bits")

**Output** — `RfamRegionsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Rfam accession of the family |
| <code>rfam_id</code> | <code>str</code> | required | Rfam family ID |
| <code>rfam_release</code> | <code>str &#124; None</code> | <code>None</code> | Rfam release of the regions |
| <code>total_regions</code> | <code>int</code> | required | Regions the family has across all sequences |
| <code>matched_regions</code> | <code>int</code> | required | Regions left after the input filters |
| <code>truncated</code> | <code>bool</code> | required | Whether matched regions exceeded max_regions |
| <code>regions</code> | <code>list[RfamRegion]</code> | <code>[]</code> | Matched regions, at most max_regions of them |

**`RfamRegion`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_accession</code> | <code>str</code> | required | Versioned accession, e.g. 'AM181176.4' |
| <code>bit_score</code> | <code>float</code> | required | Infernal bit score of the hit |
| <code>start</code> | <code>int</code> | required | First position of the hit (1-indexed, inclusive) |
| <code>end</code> | <code>int</code> | required | Last position of the hit (1-indexed, inclusive) |
| <code>strand</code> | <code>Literal['+', '-']</code> | required | Strand of the hit on the sequence |
| <code>description</code> | <code>str</code> | required | Description of the sequence holding the hit |
| <code>species</code> | <code>str</code> | required | Species name of the sequence |
| <code>taxid</code> | <code>int</code> | required | NCBI taxonomy ID of the species |

TwoAYGGAY (RF01731), Rfam 15.1
40 of 1499 regions are in SBW25
  AM181176.4 1720658-1720784 (+)  66.1 bits
  AM181176.4 1657857-1657984 (+)  65.7 bits
  AM181176.4 2296975-2297101 (-)  65.4 bits
  AM181176.4 2681329-2681456 (+)  65.3 bits
  AM181176.4 2536408-2536529 (-)  64.5 bits


#### Pull a Locus with Flanks

The coordinates line up with `run_ncbi_efetch`'s `seq_start`, `seq_stop`, and `strand`, so the best-scoring hit can be fetched directly, here with 200 nt of flanking sequence on each side.

In [8]:
from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

top = max(regions_result.regions, key=lambda r: r.bit_score)
flank = 200
locus = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore",
        identifier=top.sequence_accession,
        seq_start=top.start - flank,
        seq_stop=top.end + flank,
        strand=top.strand,
    ),
    NCBIEfetchConfig(),
)
record = locus.fasta_records[0]
print(record.header)
print(f"Length: {len(record.sequence)} nt (hit {top.end - top.start + 1} nt + {2 * flank} nt flanks)")

AM181176.4:1720458-1720984 Pseudomonas fluorescens SBW25 complete genome
Length: 527 nt (hit 127 nt + 400 nt flanks)


### `run_rfam_family`

`run_rfam_family` fetches a family by accession or ID. It returns the curation record (type, authors, seed and structure sources, counts, clan, and bit-score cutoffs) and the consensus secondary structure with its reference sequence, read from the seed alignment. The full Stockholm seed alignment is returned when `include_seed_alignment` is set.

In [9]:
from proto_tools import (
    RfamFamilyConfig,
    RfamFamilyInput,
    run_rfam_family,
)

In [10]:
# Display docs
display_api_reference("rfam", "input", "run_rfam_family")

# The family ID works as well as the accession
family_inputs = RfamFamilyInput(family="TwoAYGGAY")

**Input** — `RfamFamilyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>family</code> | <code>str</code> | required | Rfam accession (e.g. 'RF01731') or family ID (e.g. 'TwoAYGGAY') |

In [11]:
# Display config docs
display_api_reference("rfam", "config", "run_rfam_family")

# Also return the full seed alignment | see docs above for all fields
family_config = RfamFamilyConfig(include_seed_alignment=True)

**Config** — `RfamFamilyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>include_seed_alignment</code> | <code>bool</code> | <code>False</code> | Return the full Stockholm seed alignment, not only its consensus lines |

In [12]:
# Run the fetch
family_result = run_rfam_family(family_inputs, family_config)

In [13]:
# Display output docs
display_api_reference("rfam", "output", "run_rfam_family")

print(f"{family_result.accession} {family_result.rfam_id}: {family_result.description} ({family_result.rna_type})")
print(f"Seed {family_result.num_seed}, full {family_result.num_full}, species {family_result.num_species}, clan {family_result.clan_id}")
print(f"Gathering cutoff: {family_result.gathering_cutoff} bits")
print()
print(family_result.consensus_structure)
print(family_result.consensus_sequence)

**Output** — `RfamFamilyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Rfam accession |
| <code>rfam_id</code> | <code>str</code> | required | Rfam family ID |
| <code>description</code> | <code>str</code> | required | One-line family description |
| <code>rna_type</code> | <code>str</code> | required | Rfam type annotation (e.g. 'Cis-reg', 'Gene; snRNA') |
| <code>comment</code> | <code>str &#124; None</code> | <code>None</code> | Curator comment, when present |
| <code>author</code> | <code>str</code> | required | Family authors |
| <code>seed_source</code> | <code>str</code> | required | Where the seed alignment came from |
| <code>structure_source</code> | <code>str &#124; None</code> | <code>None</code> | Where the consensus structure came from |
| <code>num_seed</code> | <code>int</code> | required | Sequences in the seed alignment |
| <code>num_full</code> | <code>int</code> | required | Annotated regions across all sequences |
| <code>num_species</code> | <code>int</code> | required | Species with at least one annotated region |
| <code>clan_accession</code> | <code>str &#124; None</code> | <code>None</code> | Accession of the clan |
| <code>clan_id</code> | <code>str &#124; None</code> | <code>None</code> | ID of the clan |
| <code>gathering_cutoff</code> | <code>float</code> | required | Bit-score threshold for membership |
| <code>trusted_cutoff</code> | <code>float</code> | required | Lowest bit score of a true member |
| <code>noise_cutoff</code> | <code>float</code> | required | Highest bit score of a non-member |
| <code>rfam_release</code> | <code>str</code> | required | Rfam release the record comes from |
| <code>rfam_release_date</code> | <code>str</code> | required | Date of that release |
| <code>consensus_structure</code> | <code>str</code> | required | Consensus secondary structure (#=GC SS_cons), WUSS notation |
| <code>consensus_sequence</code> | <code>str</code> | required | Reference consensus sequence (#=GC RF), column-aligned |
| <code>seed_alignment</code> | <code>str &#124; None</code> | <code>None</code> | Full Stockholm seed alignment, when requested |

RF01731 TwoAYGGAY: TwoAYGGAY RNA (Cis-reg)
Seed 210, full 1499, species 222, clan Csr_Rsm_clan
Gathering cutoff: 43.0 bits

:::::::((((((.-.((.(((.((((((.((((.((((---------...(-..((((<<.<<<.<<<<<____>>>>>>>>>>,<<<<.<<<.<<<<<<<<____>>>>.>>>>.>>>>>>>,,)))))-...--------------------.))))))))..))))))..)))-)))))))):::::::
UauggaAgcagcu.U.ag.aua.cUcuua.aauu.uucccuuuAaacA...CA..Gcaaug.ccC.ggaCAuGGAUGuccGggcaAGucg.uaa.acgcGCCaUGGAuGGC.gcgu.uuacgaCGGuugCGU...CAaguauaaAaAcccuuaau.ggaaaauu..uaagAg..uauCcuagcugcGuAguuu


## Export Results

Regions export to JSON, TSV, or CSV, and the family exports to JSON or, when the seed alignment was requested, to Stockholm (`sto`).

In [14]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

regions_result.export("rfam_regions_sbw25", export_path=output_dir, file_format="tsv")
print(f"Exported to {output_dir / 'rfam_regions_sbw25.tsv'}")

family_result.export("rfam_family_rf01731", export_path=output_dir, file_format="sto")
print(f"Exported to {output_dir / 'rfam_family_rf01731.sto'}")

Exported to example_output/rfam_regions_sbw25.tsv
Exported to example_output/rfam_family_rf01731.sto
